# Voit-on venir la faillite ?

**Projet du cours « Python pour la data science » (ENSAE, 2A)** – Lucas Poignant, Olivier Caffier, Claudia Aguado

**Problématique.** Les entreprises défaillantes présentent-elles des signaux financiers dégradés dans les années
qui précèdent leur défaillance, et ces signaux sont-ils assez nets pour être repérés à l'avance ?

Ce notebook ne contient que l'argumentation, les tableaux et les graphiques : tous les calculs sont faits par
les fonctions du dossier `src/`. Il relit les résultats intermédiaires enregistrés sur le stockage S3 du SSP Cloud
(lecture publique, sans clé). La chaîne complète, depuis les données brutes, est décrite dans le `README.md`.

In [ ]:
# Fonctions du projet (dossier src/) et réglages d'affichage
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import config
from src.analyse import graphiques, heterogeneite, portrait, regles_alerte, silence_comptable, temoins, trajectoires
from src.collecte import contexte
from src.nettoyage import base_analyse, nettoyage_bodacc

pd.set_option("display.max_columns", 30)

## Introduction

In [ ]:
df_serie = contexte.lire_serie_defaillances()
fig, ax = graphiques.tracer_courbe(
    df_serie, "date", "nombre_defaillances",
    "Défaillances d'entreprises en France, par trimestre",
    "Nombre de défaillances (données brutes)",
    f"INSEE, série {config.IDBANK_DEFAILLANCES} (par date de jugement)",
)
debut_covid = pd.Timestamp(f"{config.ANNEE_DEBUT_JUGEMENTS}-01-01")
fin_covid = pd.Timestamp(f"{config.ANNEE_FIN_PERIODE_COVID}-12-31")
graphiques.surligner_periode(ax, debut_covid, fin_covid, "période Covid")

> **À rédiger par Claudia.** Points à couvrir :
> - le contexte : l'évolution des défaillances, le creux de la période Covid et le rattrapage ensuite ;
> - l'enjeu : pourquoi repérer tôt une entreprise en difficulté ;
> - la problématique et le plan du rapport.

## Partie 1 – Les données

Le projet croise trois sources publiques, reliées entre elles par le numéro SIREN de l'entreprise.

| Source | Accès | Ce qu'elle apporte |
|---|---|---|
| **BODACC** (DILA) | API OpenDataSoft, export parquet année par année | les jugements d'ouverture de procédure collective (sauvegarde, redressement, liquidation) : *qui* fait défaut et *quand* |
| **Bilans INPI** (données « Signaux Faibles ») | fichier parquet, data.gouv.fr | les postes de la liasse fiscale (chiffre d'affaires, résultat, capitaux propres, dettes, total du bilan…) à partir desquels on calcule les ratios financiers |
| **SIRENE** (INSEE) | stocks parquet des unités légales et des établissements, data.gouv.fr | les caractéristiques de chaque entreprise : activité (section NAF), date de création, effectif, catégorie juridique, département du siège |

Le BODACC fournit l'événement à expliquer, les bilans fournissent les signaux observés avant cet événement, et SIRENE permet de décrire les entreprises et de les comparer à des entreprises semblables (appariement du 1.4). La série trimestrielle des défaillances de l'INSEE, utilisée en introduction, sert seulement de point de comparaison pour vérifier que nos jugements BODACC sont cohérents avec les chiffres officiels (1.1).

**Période d'étude.** On étudie les entreprises dont le premier jugement d'ouverture date de **2020 à 2024**. Les données sont collectées sur une période plus large : le BODACC de 2017 à 2025, pour repérer les entreprises déjà en procédure avant 2020 et pour récupérer les jugements de fin 2024 publiés début 2025 ; les bilans clos de 2015 à 2025, pour disposer de quatre années de comptes (N-4) avant un jugement de début 2020. On s'arrête aux jugements de 2024 car les comptes et les annonces de 2025 sont encore incomplets.

**La période Covid.** Les jugements de 2020 et 2021 sont atypiques : les aides publiques (prêts garantis par l'État, chômage partiel, reports de charges) ont fortement réduit le nombre de défaillances, qui ont ensuite rattrapé leur niveau d'avant-crise. Plutôt que de retirer ces deux années, on les garde et on ajoute une colonne `periode` qui distingue la période **Covid (2020-2021)** de l'**après-Covid (2022-2024)**. Les principaux graphiques sont tracés séparément pour les deux périodes, ce qui permet de vérifier si les signaux d'alerte sont les mêmes quand les défaillances sont retardées par les aides.

### 1.1 Les défaillances (BODACC) comparées à la série officielle

In [ ]:
df_jugements = portrait.lire_jugements_periode()
display(nettoyage_bodacc.comparer_avec_insee(df_jugements))

> **À rédiger par Lucas.** Points à couvrir :
> - le choix du premier jugement d'ouverture par SIREN et les expressions régulières utilisées ;
> - l'écart entre nos jugements et la série INSEE.

### 1.2 Les choix de nettoyage des bilans

> **À rédiger par Olivier.** Points à couvrir :
> - chaque choix de nettoyage et sa justification : types de bilan gardés, doublons, postes absents mis à 0, définition des ratios, winsorisation à 1 % et 99 % ;
> - les limites : pas de liquidité pour les bilans simplifiés, comptes confidentiels sans compte de résultat.

### 1.3 De tous les jugements à la base d'analyse : l'entonnoir

Le tableau suit les entreprises de la première étape de collecte jusqu'à la base d'analyse. La première ligne compte tous les premiers jugements d'ouverture du BODACC collecté de 2017 à 2025 (402 883). On ne garde ensuite que les entreprises jugées de 2020 à 2024, soit **203 679 entreprises** : c'est la base 100 de la colonne `part_periode_pct`.

Les deux filtres suivants retirent très peu d'entreprises : 203 515 sont retrouvées dans SIRENE (99,9 %) et 203 127 ont une section NAF (99,7 %). Ils sont nécessaires car la section NAF sert ensuite à l'appariement et à la partie 4.

La perte principale vient des comptes annuels : seules **61 722 entreprises (30 %)** ont au moins un bilan clos avant leur jugement dans les comptes INPI collectés (exercices 2015 à 2025). Deux raisons l'expliquent. Les entrepreneurs individuels et les micro-entreprises ne sont pas tenus de déposer des comptes, alors qu'ils représentent une part importante des défaillances. Parmi les sociétés, beaucoup de petites structures ne déposent pas leurs comptes malgré l'obligation, la sanction étant rarement appliquée.

Enfin, on exige un bilan clos dans les quatre ans qui précèdent le jugement. L'horizon est glissant : N-1 désigne un bilan clos dans les 12 mois avant le jugement, N-2 entre 12 et 24 mois, etc. Ce choix évite de mélanger, sous une même « année N-1 » civile, des bilans clos un mois et vingt-trois mois avant le jugement. **47 552 entreprises (23 %)** ont au moins un bilan entre N-1 et N-4. Les 14 170 autres avaient un dernier bilan plus ancien : elles avaient cessé de publier bien avant leur jugement, ce qui annonce le silence comptable étudié en partie 5.

La base d'analyse n'est donc pas représentative de l'ensemble des défaillances : elle porte sur les sociétés qui publient leurs comptes, plus grandes et plus structurées que la moyenne. Nos conclusions valent pour ce champ.

In [ ]:
df_defaillantes, df_bilans_defaillantes = temoins.lire_base_defaillantes()
nombre_jugements_total = base_analyse.compter_jugements_total()
display(base_analyse.construire_entonnoir(nombre_jugements_total, df_defaillantes))

### 1.4 Le groupe témoin

**Pourquoi un groupe témoin.** Un ratio qui se dégrade avant le jugement n'est un signal d'alerte que s'il se dégrade moins chez des entreprises comparables qui ne font pas défaut. Sans point de comparaison, on ne peut pas séparer une dégradation propre aux défaillantes d'un choc commun à toutes les entreprises (Covid, hausse des prix et des taux en 2022-2023).

**La méthode.** Pour chaque défaillante, on retient son bilan de référence, le plus récent entre N-1 et N-4. On définit une « case » par trois critères : la section NAF, la tranche de total du bilan (moins de 100 k€, 100 à 500 k€, 500 k€ à 2 M€, plus de 2 M€) et l'année de clôture du bilan de référence. Dans chaque case, on tire au hasard un témoin parmi les entreprises qui n'apparaissent jamais dans le BODACC collecté (2017-2025) et dont la section et le total du bilan sont connus. Ce réservoir compte 5,79 millions de bilans. Les principaux choix sont les suivants :
- **taille mesurée par le total du bilan** : le chiffre d'affaires est masqué dans 35 % des bilans N-1 (comptes confidentiels) alors que le total du bilan est toujours publié, et l'effectif SIRENE est la valeur actuelle, pas celle de l'année du bilan. Les seuils sont des valeurs rondes proches des quantiles du total du bilan des défaillantes à N-1 (médiane d'environ 525 k€). Ils découpent les bilans des défaillantes en tranches de taille comparable (entre 20 % et 34 % à N-1), sauf la dernière, plus petite (18 % à N-1, 12 % sur l'ensemble des horizons) ;
- **même année de clôture** : les deux bilans comparés ont été établis dans la même conjoncture ;
- **un témoin par défaillante**, et une entreprise ne peut être témoin qu'une fois ;
- **tirage reproductible** : le réservoir est trié puis mélangé avec une graine fixée (2026), donc relancer le code redonne les mêmes paires ;
- **date de jugement fictive** : chaque témoin reçoit la date de jugement de sa défaillante. Ses horizons N-1 à N-4 et sa période (Covid ou après Covid) sont calculés exactement comme pour elle.

**Lecture des tableaux.** Le premier tableau montre que l'appariement ne coûte presque rien. Il part de 47 573 défaillantes ayant un bilan entre N-1 et N-4 : c'est 21 de plus que dans l'entonnoir du 1.3, car ce tableau ne retire pas les entreprises absentes de SIRENE. 28 défaillantes sont écartées faute de section NAF ou de total du bilan, et **les 47 545 restantes ont toutes trouvé un témoin** : le réservoir est assez grand pour qu'aucune case ne soit à court de candidats.

Le second tableau compte les bilans de la base appariée (240 404 bilans au total) par période et par horizon. Le fait marquant est le déséquilibre à N-1 : **10 544 bilans de défaillantes contre 33 974 bilans de témoins**, soit environ 22 % des défaillantes contre 71 % des témoins. Autrement dit, les défaillantes ont à N-1 environ trois fois moins de bilans que leurs témoins, et ce rapport est proche dans les deux périodes : 28 % en période Covid (2 606 bilans contre 9 379) et 32 % après (7 938 contre 24 595). À N-4, il remonte à environ 94 %. Les défaillantes cessent donc de publier à l'approche du jugement. Une partie de l'écart est mécanique : un témoin n'est tenu d'avoir un bilan que l'année de référence de sa défaillante, souvent N-2 ou N-3. Ce silence est étudié en partie 5. Il implique dès maintenant que les comparaisons à N-1 de la partie 3 portent sur une minorité de défaillantes, qui n'est peut-être pas représentative.

In [ ]:
df_reference = temoins.choisir_bilans_reference(df_defaillantes, df_bilans_defaillantes)
df_paires = silence_comptable.lire_paires()
display(temoins.construire_entonnoir_appariement(df_defaillantes, df_reference, df_paires))

df_base = trajectoires.lire_base_appariee()
display(temoins.comparer_groupes(df_base))

**Limites de l'appariement.**
- Le total du bilan de la défaillante est mesuré l'année de référence, alors que sa situation peut déjà être dégradée. Le témoin est donc apparié sur une taille déjà affectée. De plus, les tranches sont larges : à l'intérieur d'une case, les deux groupes peuvent avoir des tailles différentes.
- On n'apparie ni sur l'âge, ni sur la forme juridique, ni sur le département. Les écarts de ratios peuvent donc refléter en partie ces différences, en particulier l'âge, qui joue fortement sur le risque de défaillance (partie 2.2).
- Les témoins sont des entreprises qui publient leurs comptes, donc plutôt mieux organisées que la moyenne des entreprises non défaillantes. Certains ont pu cesser leur activité sans procédure collective (dissolution amiable), et ceux appariés à des jugements de 2024 sont observés peu de temps après : quelques-uns feront peut-être défaut après 2025.
- Avec un témoin par défaillante, la base contient 50 % de défaillantes, contre quelques pour mille dans la population réelle. Les taux de fausses alertes de la partie 6 ne sont donc pas transposables tels quels.

## Partie 2 – Portrait des entreprises défaillantes

In [ ]:
# Lit SIRENE en ligne : compter 2 à 5 minutes
df_comptages = portrait.compter_entreprises_et_defaillances()
periode = f"{config.ANNEE_DEBUT_JUGEMENTS}-{config.ANNEE_FIN_JUGEMENTS}"
titre_axe_taux = "Défaillances pour 1 000 sociétés"
source_portrait = "BODACC, INSEE (SIRENE) ; calculs des auteurs"

### 2.1 Selon le secteur

In [ ]:
df_taux_secteurs = portrait.calculer_taux(df_comptages, "libelle_section")
df_taux_secteurs = df_taux_secteurs.sort_values("taux_pour_mille", ascending=False)
fig, ax = graphiques.tracer_barres_horizontales(
    df_taux_secteurs, "libelle_section", "taux_pour_mille",
    f"Taux annuel de défaillance par secteur, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire du graphique par secteur.

### 2.2 Selon l'âge

In [ ]:
df_taux_ages = portrait.calculer_taux(df_comptages, "tranche_age")
fig, ax = graphiques.tracer_barres_horizontales(
    df_taux_ages, "tranche_age", "taux_pour_mille",
    f"Taux annuel de défaillance selon l'âge de la société, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire du graphique par âge.

### 2.3 Selon la taille

In [ ]:
df_taux_tailles = portrait.calculer_taux(df_comptages, "groupe_effectifs")
fig, ax = graphiques.tracer_barres_horizontales(
    df_taux_tailles, "groupe_effectifs", "taux_pour_mille",
    f"Taux annuel de défaillance selon l'effectif salarié, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire du graphique par taille.

### 2.4 Selon le département

In [ ]:
df_taux_departements = portrait.calculer_taux(df_comptages, "departement")
gdf_departements = portrait.preparer_carte(df_taux_departements)
fig, ax = graphiques.tracer_carte(
    gdf_departements, "taux_pour_mille",
    f"Taux annuel de défaillance par département, {periode}", titre_axe_taux, source_portrait,
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire de la carte.

### 2.5 Les types de procédure

In [ ]:
fig, ax = graphiques.tracer_barres_empilees(
    portrait.repartir_procedures(df_jugements),
    "Répartition des premiers jugements par type de procédure", "Part des jugements (%)",
    "BODACC ; calculs des auteurs",
)

> **À rédiger par Lucas.** Points à couvrir :
> - commentaire de la répartition des procédures.

## Partie 3 – Les trajectoires des ratios de N-4 à N-1

### 3.1 Défaillantes contre témoins

In [ ]:
figures_trajectoires = trajectoires.tracer_trajectoires_par_periode(
    df_base, config.RATIOS_TRAJECTOIRES, "Trajectoires des ratios"
)

In [ ]:
# Médianes de chaque ratio, par groupe et par horizon (les deux périodes ensemble)
df_quartiles = trajectoires.calculer_quartiles(df_base, config.RATIOS_TRAJECTOIRES, ["groupe"])
display(trajectoires.presenter_medianes(df_quartiles, ["groupe"]).round(3))

> **À rédiger par Olivier.** Points à couvrir :
> - quel ratio décroche en premier, et à quel horizon (critère de validation de l'étape 8) ;
> - les différences entre la période Covid et l'après-Covid.

### 3.2 Vérification : les entreprises présentes aux quatre horizons

In [ ]:
df_base_complete = trajectoires.garder_entreprises_completes(df_base)
figures_completes = trajectoires.tracer_trajectoires_par_periode(
    df_base_complete, config.RATIOS_TRAJECTOIRES, "Trajectoires des ratios, entreprises présentes de N-4 à N-1"
)

> **À rédiger par Olivier.** Points à couvrir :
> - ce que la vérification confirme ou nuance dans la lecture du 3.1.

### 3.3 Les rangs sectoriels à N-1 et N-2

In [ ]:
df_base_rangs = trajectoires.ajouter_rangs(df_base, trajectoires.lire_rangs_sectoriels())
display(trajectoires.resumer_rangs(df_base_rangs))

> **À rédiger par Olivier.** Points à couvrir :
> - la position des défaillantes dans leur secteur, et pourquoi les témoins sont proches de 50.

### 3.4 Les situations extrêmes

In [ ]:
display(trajectoires.compter_situations_extremes(df_base, ["groupe"]))

> **À rédiger par Olivier.** Points à couvrir :
> - capitaux propres négatifs et trésorerie quasi nulle : à quel horizon l'écart se creuse.

## Partie 4 – Des trajectoires différentes selon les entreprises

In [ ]:
df_sous_groupes = heterogeneite.ajouter_sous_groupes(df_base, heterogeneite.construire_sous_groupes())

### 4.1 Selon le grand secteur

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "grand_secteur")

> **À rédiger par Olivier.** Points à couvrir :
> - un paragraphe d'interprétation (critère de validation de l'étape 9), en citant les effectifs.

### 4.2 Selon l'âge au jugement

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "classe_age")

> **À rédiger par Olivier.** Points à couvrir :
> - un paragraphe d'interprétation (critère de validation de l'étape 9), en citant les effectifs.

### 4.3 Selon la taille

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "taille_appariement")

> **À rédiger par Olivier.** Points à couvrir :
> - un paragraphe d'interprétation (critère de validation de l'étape 9), en citant les effectifs.

### 4.4 Selon le type de procédure

In [ ]:
heterogeneite.afficher_sous_groupe(df_sous_groupes, "type_procedure")

> **À rédiger par Olivier.** Points à couvrir :
> - un paragraphe d'interprétation (critère de validation de l'étape 9), en citant les effectifs.

## Partie 5 – Le silence comptable

In [ ]:
df_toutes_defaillantes = silence_comptable.lire_defaillantes()
display(silence_comptable.decrire_population_totale(df_toutes_defaillantes, df_paires))

In [ ]:
df_grille = silence_comptable.construire_grille(df_paires, df_toutes_defaillantes)
df_grille = silence_comptable.ajouter_statut(df_grille, df_base)
display(silence_comptable.calculer_taux_silence(df_grille, ["groupe"]))
figures_silence = silence_comptable.tracer_silence_par_periode(df_grille)

In [ ]:
# Biais de sélection : ratios à N-2 des défaillantes, selon qu'elles publient encore à N-1
display(silence_comptable.comparer_publiantes_et_silencieuses(df_base))

> **À rédiger par Olivier.** Points à couvrir :
> - l'ampleur du silence comptable avant le jugement, défaillantes contre témoins ;
> - le biais de sélection que ce silence crée pour la partie 3 (critère de validation de l'étape 10).

## Partie 6 – Des règles d'alerte simples

In [ ]:
df_alertes = regles_alerte.ajouter_alertes(df_base.copy())
display(regles_alerte.calculer_couverture(df_alertes))
display(regles_alerte.tableau_performances(df_alertes))
display(regles_alerte.tableau_performances_par_periode(df_alertes))

> **À rédiger par Lucas.** Points à couvrir :
> - les taux de détection et de fausses alertes à N-1 et N-2 ;
> - pourquoi les fausses alertes seraient plus nombreuses en population réelle (critère de validation de l'étape 11).

## Conclusion et limites

> **À rédiger par Claudia.** Points à couvrir :
> - la réponse à la problématique ;
> - les limites : silence comptable et biais de sélection, appariement, valeurs actuelles des effectifs dans SIRENE, période Covid, comptes récents pas encore déposés.